In [16]:
import requests
import urllib.request
import time
import os
import warnings
import re

from bs4 import BeautifulSoup

import csv
import os.path

In [17]:
def generateFileCSV(listHasil,csvName1):

	csvFolrder = "./"
	csvName = csvName1

	if os.path.exists(csvName):

		f = open(csvName, 'a', newline='\n')
		print(f)
		print("ada")
		w = csv.writer(f)

	else:

		f = open(csvName, 'w', newline='\n')
		print(f)
		print("tidak ada")

		w = csv.writer(f)
		w.writerow(("terdakwa", "penuntut_umum", "nomor", "tingkat_proses", "klasifikasi", "kata_kunci", "tahun", "tanggal_register",
              "lembaga_peradilan", "jenis_lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar",
              "amar_lainnya", "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan", "kaidah", "abstrak", "url"))

	# menulis file csv
	for s in listHasil:
		w.writerow(s)

	f.close()
	berhasil = "\nCreate Csv file Berhasil\n"
	return berhasil

In [18]:
# def generateMeta(urlMeta):

#     url = str(urlMeta).strip()
#     #url = 'https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedca28e81ec25e8cb4313634373336.html'
#     response = requests.get(url, verify=False, timeout=30)
#     #print(url)
#     print(response)

#     soup = BeautifulSoup(response.text, 'html.parser')
#     cleanTags = re.compile('<.*?>')

#     listMetaHead = ["terdakwa", "penuntut_umum", "nomor", "tingkat_proses", "klasifikasi", "kata_kunci", "tahun", "tanggal_register",
#               "lembaga_peradilan", "jenis_lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar",
#               "amar_lainnya", "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan", "kaidah", "abstrak", "url"]

#     # initialisasi
#     listMeta = []
#     #for i in range(len(listMetaHead)):
#         #listMeta.append("")

    
#     rowsMETA1 = soup.find("ul", {"class": "portfolio-meta nobottommargin"}).find("table").findAll("tr")
#     rowsMETA2 = soup.findAll("ul", {"class": "portfolio-meta nobottommargin"})

#     #print('-------------------------')
#     for row in rowsMETA1:
#         coll = row.findAll("td")

#         cleantext2 =''
#         cleantext1 =''

#         if len(coll) > 1:
#             cleantext2 = (re.sub(cleanTags, '', str(coll[1]))).strip()
#             listMeta.append(cleantext2.replace('\n',' '))
#             #print(cleantext2.replace('\n',' '))

#         else:
#             cleantext1 = (re.sub(cleanTags, ' ', str(coll[0]))).strip()
#             # untuk putusan pidana akan muncul terdakwa dan penuntut umum pada meta
#             # sedangkan untuk putusan selain pidana tidak muncul

#             pidorpdt = re.search( r'(.*)/Pdt.(.*)',str(cleantext1), re.M|re.I)

#             # check dokumen putusannya pidana atau perdata
#             if pidorpdt == None:
#                 entTerdakwah = re.search( r'(.*)Terdakwa:(.*)',str(cleantext1), re.M|re.I)
#                 entPenuntut = re.search( r'Penuntut Umum:(.*)Terdakwa:',str(cleantext1), re.M|re.I)
#             else:
#                 entTerdakwah = re.search( r'(.*)Tergugat:(.*)',str(cleantext1), re.M|re.I)
#                 entPenuntut = re.search( r'Penggugat:(.*)Tergugat:',str(cleantext1), re.M|re.I)

#             if entTerdakwah == None:
#                 listMeta.append("")
#             else:
#                 listMeta.append(entTerdakwah.group(2))

#             if entPenuntut == None:
#                 listMeta.append("")
#             else:
#                 listMeta.append(entPenuntut.group(1))

#             #print(coll[0])
#             #print(entTerdakwah.group(2))
#             #print(entPenuntut.group(1))

#     urlDL = rowsMETA2[1].findAll("li")
#     urlDLStr = str(urlDL[4])
#     listMeta.append(urlDLStr[urlDLStr.find("https"):urlDLStr.find('">')])
#     #print(urlDLStr[urlDLStr.find("https"):urlDLStr.find('">')])
#     #print(listMeta)

#     return listMeta

In [19]:
def generateMeta(urlMeta):
    url = str(urlMeta).strip()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
        "Referer": "https://putusan3.mahkamahagung.go.id/",
    }

    response = requests.get(
        url,
        headers=headers,
        verify=False,
        timeout=30
    )

    print(response)

    if response.status_code != 200:
        raise RuntimeError(
            f"HTTP {response.status_code} untuk URL: {url}"
        )

    soup = BeautifulSoup(response.text, "html.parser")
    cleanTags = re.compile("<.*?>")

    meta_section = soup.find(
        "ul",
        {"class": "portfolio-meta nobottommargin"}
    )

    if meta_section is None:
        raise RuntimeError(
            f"Struktur halaman tidak ditemukan untuk URL: {url}"
        )

    table = meta_section.find("table")

    if table is None:
        raise RuntimeError(
            f"Tabel metadata tidak ditemukan untuk URL: {url}"
        )

    rowsMETA1 = table.findAll("tr")
    listMeta = []

    for row in rowsMETA1:
        coll = row.findAll("td")

        if len(coll) > 1:
            cleantext = re.sub(cleanTags, "", str(coll[1])).strip()
            listMeta.append(cleantext.replace("\n", " "))
        elif len(coll) == 1:
            cleantext = re.sub(cleanTags, " ", str(coll[0])).strip()

            if re.search(r"/Pdt\.", cleantext, re.I):
                terdakwa_match = re.search(
                    r"(.*)Tergugat:(.*)",
                    cleantext,
                    re.I
                )
                penuntut_match = re.search(
                    r"Penggugat:(.*)Tergugat:",
                    cleantext,
                    re.I
                )
            else:
                terdakwa_match = re.search(
                    r"(.*)Terdakwa:(.*)",
                    cleantext,
                    re.I
                )
                penuntut_match = re.search(
                    r"Penuntut Umum:(.*)Terdakwa:",
                    cleantext,
                    re.I
                )

            listMeta.append(
                terdakwa_match.group(2).strip()
                if terdakwa_match else ""
            )
            listMeta.append(
                penuntut_match.group(1).strip()
                if penuntut_match else ""
            )

    download_link = soup.select_one(
        'a[href*="/download_file/"][href*="/pdf/"]'
    )

    if download_link is None:
        raise RuntimeError(
            f"Link PDF tidak ditemukan untuk URL: {url}"
        )

    listMeta.append(download_link.get("href"))

    return listMeta

In [20]:
def main():

    warnings.filterwarnings('ignore')

    #folderListURL = pathFile
    # fileListURL = "./listURLPerceraianPASBY.txt"
    fileListURL = "./listURLPerceraianPAJAKPUS1.txt"
    fileMetaCSV = "./metaPerceraianPAJAKPUS1.csv"
    listHasil =[]

    startTime = time.time()
    openfileListURL = open(fileListURL, "r", encoding='UTF8')
    bacaListURL = openfileListURL.readlines()

    i = 1
    for barisURL in bacaListURL:
        try:
            #print(str(barisURL))
            hasil = generateMeta(str(barisURL))
            listHasil.append(hasil)
            time.sleep(3)
            print("======= ROW HASIL =======",i)
            #print(hasil)
        except Exception as e:
            # print(f"Error Get Meta Inf, {e}")
            print(f"Error pada URL ke {i}: {e}")
        i=i+1

    createFile = generateFileCSV(listHasil,fileMetaCSV)

    openfileListURL.close()
    endTime = time.time()
    #print(listHasil)
    print('Time Processing : ', endTime-startTime, ' Second')

main();



<Response [200]>
Error pada URL ke 1: Link PDF tidak ditemukan untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b4b7ddc34a65f313730393536.html
<Response [200]>
Error pada URL ke 2: Link PDF tidak ditemukan untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b4b061708a65f313730393536.html
<Response [200]>
Error pada URL ke 3: Link PDF tidak ditemukan untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b4a68a734801f313730393535.html
<Response [200]>
Error pada URL ke 4: Link PDF tidak ditemukan untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b4a033e44952a313730393534.html
<Response [403]>
Error pada URL ke 5: HTTP 403 untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b491d85c09d78313730393533.html
<Response [403]>
Error pada URL ke 6: HTTP 403 untuk URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaedff9b49590e10bfff313730393533.html
<Response [403]>
Error pada UR

KeyboardInterrupt: 